# 04 Test year
Final, one-time evaluation of every model on the test year (Jul 2025 to Jun 2026), data no model has seen and that was never used for any choice.

Settings are fixed from the validation year:
- All trained models predict with **day-ahead weather forecasts**
- LightGBM: large (63 leaves, 800 trees, learning rate 0.03), without the `month` feature
- Walk-forward: retrain monthly on all earlier data (expanding window)

In [2]:
# Setup
import sys
sys.path.append("../src")                       # Python finds src/forecasting.py

import pandas as pd
import lightgbm as lgb
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, QuantileRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from forecasting import (FEATURES, LAGS, WEATHER, TEST_START, TEST_END,
                         load_model_data, evaluate, walk_forward, with_forecast_weather)

df = load_model_data()
print(f"Rows usable for modelling: {len(df):,}")
# Every test-year row needs a weather forecast; this should print 0
print("Forecast weather gaps in the test year:",
      df.loc[df["date"] >= TEST_START, ["akl_temp_fc", "mnw_wind_fc"]].isna().sum().sum())

Rows usable for modelling: 59,424
Forecast weather gaps in the test year: 0


## Models (same definitions as seen in notebooks 02 and 03)

In [3]:
# Naive models: no training needed
def naive(train, test):
    return test["lag_1d"]          # yesterday, same trading period

def weekly_naive(train, test):
    return test["lag_7d"]          # last week, same trading period


def linear_pipeline(regressor):
    """Feature preparation for the linear models (as in notebook 02):
    one-hot encode tp/dow/month, standardise lags and weather, pass is_holiday through."""
    return make_pipeline(
        ColumnTransformer([
            ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"), ["tp", "dow", "month"]),
            ("scale", StandardScaler(), LAGS + WEATHER),
            ("holiday", "passthrough", ["is_holiday"]),
        ]),
        regressor,
    )


def make_fit_predict(model, features=FEATURES):
    """Wrap a model so walk_forward can call it: fit on train, predict test."""
    def fit_predict(train, test):
        model.fit(train[features], train["price"])
        return model.predict(test[features])
    return fit_predict


# Final LightGBM settings, chosen on the validation year (notebook 03)
FINAL_LGBM_FEATURES = list(pd.Index(FEATURES).drop("month"))
FINAL_LGBM_PARAMS = dict(num_leaves=63, n_estimators=800, learning_rate=0.03)

ols_model = linear_pipeline(LinearRegression())
lad_model = linear_pipeline(QuantileRegressor(quantile=0.5, alpha=0, solver="highs"))
lgbm_model = lgb.LGBMRegressor(objective="l1", random_state=42, verbose=-1, **FINAL_LGBM_PARAMS)

## Run the fast models

In [4]:
test_preds = pd.DataFrame({
    "Model 0 Naive (yesterday)": walk_forward(df, naive, TEST_START, TEST_END),
    "Model 1 Weekly naive (last week)": walk_forward(df, weekly_naive, TEST_START, TEST_END),
    "Model 2a Linear (least squares)": walk_forward(
        df, with_forecast_weather(make_fit_predict(ols_model)), TEST_START, TEST_END),
    "Model 3 LightGBM": walk_forward(
        df, with_forecast_weather(make_fit_predict(lgbm_model, FINAL_LGBM_FEATURES)), TEST_START, TEST_END),
})
test_preds.to_parquet("../results/test_predictions.parquet")      # save straight away

  07/2025 done
  08/2025 done
  09/2025 done
  10/2025 done
  11/2025 done
  12/2025 done
  01/2026 done
  02/2026 done
  03/2026 done
  04/2026 done
  05/2026 done
  06/2026 done
  07/2025 done
  08/2025 done
  09/2025 done
  10/2025 done
  11/2025 done
  12/2025 done
  01/2026 done
  02/2026 done
  03/2026 done
  04/2026 done
  05/2026 done
  06/2026 done
  07/2025 done
  08/2025 done
  09/2025 done
  10/2025 done
  11/2025 done
  12/2025 done
  01/2026 done
  02/2026 done
  03/2026 done
  04/2026 done
  05/2026 done
  06/2026 done
  07/2025 done
  08/2025 done
  09/2025 done
  10/2025 done
  11/2025 done
  12/2025 done
  01/2026 done
  02/2026 done
  03/2026 done
  04/2026 done
  05/2026 done
  06/2026 done


## Run the linear (absolute-error) model (this is slower, as it goes through one LP per month)

In [5]:
test_preds["Model 2b Linear (absolute error)"] = walk_forward(
    df, with_forecast_weather(make_fit_predict(lad_model)), TEST_START, TEST_END)
test_preds.to_parquet("../results/test_predictions.parquet")      # save again, now with all five models

  07/2025 done
  08/2025 done
  09/2025 done
  10/2025 done
  11/2025 done
  12/2025 done
  01/2026 done
  02/2026 done
  03/2026 done
  04/2026 done
  05/2026 done
  06/2026 done


## Results: test year (Jul 2025 to Jun 2026)

In [6]:
test = df.loc[test_preds.index]        # the test-year rows, lined up with the predictions


def score(preds):
    """Score every column of a predictions table; one row per model."""
    return preds.apply(lambda p: evaluate(test["price"], p, test["date"])).T.round(2)


test_results = score(test_preds)
test_results.to_csv("../results/test_results.csv")
test_results

,MAE,RMSE,Spearman
Model 0 Naive (yesterday),36.48,66.97,0.38
Model 1 Weekly naive (last week),48.94,80.24,0.34
Model 2a Linear (least squares),35.39,53.40,0.51
Model 3 LightGBM,30.22,51.56,0.49
Model 2b Linear (absolute error),31.75,51.56,0.52


In [7]:
# Mean absolute error per month for every model: where does each one win or lose?
monthly_mae = (test_preds.sub(test["price"], axis=0)                       # forecast minus actual
                         .abs()                                            # absolute errors
                         .groupby(test["date"].dt.to_period("M")).mean()   # average per month
                         .round(1))
monthly_mae.index = monthly_mae.index.strftime("%m/%Y")
monthly_mae

,Model 0 Naive (yesterday),Model 1 Weekly naive (last week),Model 2a Linear (least squares),Model 3 LightGBM,Model 2b Linear (absolute error)
date,,,,,
07/2025,63.5,74.6,52.5,52.3,51.4
08/2025,36.7,54.2,32.2,28.9,29.4
09/2025,42.0,63.9,42.2,37.7,37.8
10/2025,37.5,50.5,45.2,29.3,37.6
11/2025,42.1,56.0,37.4,36.1,37.7
12/2025,19.0,36.0,31.6,18.8,24.2
01/2026,5.6,6.5,25.1,7.2,15.4
02/2026,38.2,41.7,31.8,28.4,29.7
03/2026,31.7,48.0,26.1,26.8,24.1


## what are the key findings?
1. **LightGBM is the best forecaster on the test year with MAE \$30.22/MWh, 17.2% below naive (\$36.48)**, reversing the validation result. It struggled only in the unprecedented August 2024 crisis; the test year stayed within the range of prices seen in training.
2. **Linear models struggle when prices sit near \$0** (January 2026: MAE \$15.4 to \$25.1 vs LightGBM's \$7.2), because they can predict prices the market is pinned below.
3. **Absolute error clearly beat least squares on the test year** (\$31.75 vs \$35.39), confirming it as the right loss for spiky prices.
4. **The lowest MAE is not the best ranking:** linear (absolute error) ranks each day's periods better (Spearman 0.52 vs 0.49). Which forecast earns the battery more money is tested on Day 4.